In [1]:
import pandas as pd
import numpy as np
import tifffile as tiff
from deepcell_spots.applications import Polaris
from pathlib import Path
import tensorflow as tf
import os


SG_MPP_60X = 0.10727

data_dir = Path('/mnt/deepcell_data/users/ellen/macrophages/signaling')

/home/eemerson/venvs/deepcell-spots/lib/python3.10/site-packages/keras/optimizer_v2/gradient_descent.py:102: UserWarning: The `lr` argument is deprecated, use `learning_rate` instead.
  super(SGD, self).__init__(name, **kwargs)


In [2]:
# tensorflow/GPU setup
device_indices = '1'
os.environ["CUDA_VISIBLE_DEVICES"]="{}".format(device_indices)

physical_devices = tf.config.experimental.list_physical_devices('GPU')
assert len(physical_devices) >= 1 , "GPU Configuration failed"

# IMPORTANT: without this config, the GPU will run out of memory trying to run Polaris
for device in physical_devices:
    tf.config.experimental.set_memory_growth(device, True)

In [3]:
c0_codebook = pd.read_csv(data_dir / 'spatial_genomics_barcodes/extended_panel/df_barcodes_c0.csv', index_col=0)

rounds = 20 # number of hybridizations

# run this once to ensure model is downloaded
# nuc_app = NuclearSegmentation.from_version('1.1')

# but then we want the model itself
model_dir = Path.home() / ".deepcell" / "models"
model_path = model_dir / 'NuclearSegmentation'
nuc_model = tf.keras.models.load_model(model_path)

polaris_app_c0 = Polaris(image_type='multiplex',
                         segmentation_type='nucleus',
                         segmentation_model=nuc_model,
                         decoding_kwargs={'rounds': rounds,
                                          'channels': 1,
                                          'df_barcodes': c0_codebook})

2025-09-08 14:59:00.876518: I tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-08 14:59:01.401743: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1525] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 29909 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:23:00.0, compute capability: 8.6


INFO:root:Checking for cached data
INFO:root:Checking SpotDetection-8.tar.gz against provided file_hash...
INFO:root:SpotDetection-8.tar.gz with hash a6164e48ef8872a9524b4ec6726859d7 already available.
INFO:root:Extracting /home/eemerson/.deepcell/models/SpotDetection-8.tar.gz
INFO:root:Successfully extracted /home/eemerson/.deepcell/models/SpotDetection-8.tar.gz into /home/eemerson/.deepcell/models


In [4]:
img = tiff.imread(data_dir / '20250820-EE_prim_mac_JNK-p65_LPS_paired/100_ng_mL/spatial_genomics/full_scale/cropped_regions/fov_0.tiff')
img = img.astype('float32')
img.shape

(4, 8400, 8400, 20)

In [5]:
single_channel_image = np.expand_dims(img[0], axis=0)
nuc_img = np.expand_dims(np.expand_dims(img[-1,...,0], axis=-1), axis=0)

print(single_channel_image.shape)
print(nuc_img.shape)

(1, 8400, 8400, 20)
(1, 8400, 8400, 1)


In [6]:
old_results, new_results, seg = polaris_app_c0.predict(single_channel_image,
                                        segmentation_image=nuc_img,
                                        image_mpp=SG_MPP_60X,
                                        decoding_training_kwargs={'rescue_errors': False,
                                                                  'rescue_mixed': False,
                                                                  'pred_prob_thresh': 0.95})

Validating inputs.
Predicting spot locations.


  0%|                                                                             | 0/20 [00:00<?, ?it/s]2025-09-08 15:04:03.907666: I tensorflow/stream_executor/cuda/cuda_dnn.cc:368] Loaded cuDNN version 8204
2025-09-08 15:04:06.134474: I tensorflow/stream_executor/cuda/cuda_blas.cc:1786] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
100%|███████████████████████████████████████████████████████████████████| 20/20 [50:06<00:00, 150.33s/it]


Segmenting cells.
Decoding gene identities.


/home/eemerson/venvs/deepcell-spots/lib/python3.10/site-packages/torch/__init__.py:749: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:431.)
  _C._set_default_tensor_type(t)


Training...


100%|██████████████████████████████████████████████████████████████████| 500/500 [02:41<00:00,  3.09it/s]


Estimating barcode probabilities...
Refining spot locations.
Refining spot locations.


In [7]:
def compare_dfs(df1, df2, verbose=True):
    identical = True

    # 1. Shape
    if df1.shape != df2.shape:
        if verbose:
            print(f"Shape differs: {df1.shape} vs {df2.shape}")
        identical = False

    # 2. Columns
    if not df1.columns.equals(df2.columns):
        if verbose:
            print(f"Columns differ:")
            print("Only in df1:", df1.columns.difference(df2.columns).tolist())
            print("Only in df2:", df2.columns.difference(df1.columns).tolist())
        identical = False

    # 3. Index
    if not df1.index.equals(df2.index):
        if verbose:
            print("Index differs")
        identical = False

    # 4. Dtypes
    if not (df1.dtypes == df2.dtypes).all():
        if verbose:
            print("Dtypes differ:")
            print("df1 dtypes:\n", df1.dtypes)
            print("df2 dtypes:\n", df2.dtypes)
        identical = False

    # 5. Values (per column)
    for col in df1.columns:
        s1 = df1[col]
        s2 = df2[col]

        # Align indices to avoid ValueError
        s1_aligned, s2_aligned = s1.align(s2)

        if pd.api.types.is_numeric_dtype(s1):
            mask_diff = ~((s1_aligned.fillna(np.nan) == s2_aligned.fillna(np.nan)))
        else:
            mask_diff = ~((s1_aligned == s2_aligned) | (s1_aligned.isna() & s2_aligned.isna()))

        if mask_diff.any():
            if verbose:
                print(f"Differences found in column '{col}':")
                for idx in s1_aligned.index[mask_diff]:
                    print(f"  Row {idx}: {s1_aligned.loc[idx]} != {s2_aligned.loc[idx]}")
            identical = False

    if identical and verbose:
        print("DataFrames are identical!")

    return identical

In [8]:
compare_dfs(old_results, new_results)

DataFrames are identical!


True